![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

## Write-Audit-Publish with Branches in Apache Iceberg

This notebook runs using the Docker Compose at https://github.com/tabular-io/docker-spark-iceberg. 
It's based on the [Iceberg - Integrated Audits Demo.ipynb](https://github.com/tabular-io/docker-spark-iceberg/blob/main/spark/notebooks/Iceberg%20-%20Integrated%20Audits%20Demo.ipynb) notebook. 

### SQL Authoring Note (Kotlin Kernel)

Known limitation (Kotlin kernel): JupyterLab does not apply SQL syntax highlighting inside Kotlin triple-quoted strings (`"""..."""`).

What works in this notebook:
- Use the toolbar **Format SQL** action for query formatting.
- Keep SQL in a dedicated variable and execute with `spark.sql(query)`.

```kotlin
val query = """
SELECT *
FROM nyc.taxis
""".trimIndent()

spark.sql(query).show(100, 0, false)
```


In [ ]:
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder().appName("Jupyter").getOrCreate()

spark

To be able to rerun the notebook several times, let's drop the `permits` table if it exists to start fresh.

In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS nyc
""".trimIndent()).show(100, 0, false)

In [ ]:
spark.sql("""
DROP TABLE IF EXISTS nyc.permits
""".trimIndent()).show(100, 0, false)

# Load NYC Film Permits Data

For this demo, we will use the [New York City Film Permits dataset](https://www.kaggle.com/datasets/new-york-city/nyc-filming-permits) available on Kaggle as a CSV snapshot (about 40k rows from 2017). Download it with the Kaggle CLI and place it at `/home/iceberg/data/nyc_film_permits.csv` to run this notebook.

We'll save the sample dataset into an iceberg table called `permits`.

In [ ]:
val permits_df = spark.read()
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/home/iceberg/data/nyc_film_permits.csv")
permits_df.write().saveAsTable("nyc.permits")

Taking a quick peek at the data, you can see that there are a number of permits for different boroughs in New York.

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits
GROUP BY borough
""".trimIndent()).show(100, 0, false)

# The Setup

Tables by default are not configured to allow integrated audits, therefore the first step is enabling this by setting the `write.wap.enabled` table metadata property to `true`

In [ ]:
spark.sql("""
ALTER TABLE nyc.permits
SET TBLPROPERTIES (
    'write.wap.enabled'='true'
)
""".trimIndent()).show(100, 0, false)

We create a branch for the work we want to do. This is a copy-on-write branch, so "free" until we start making changes (and "cheap" thereafter) since only data that's changed needs to be written. 

In [ ]:
spark.sql("""
ALTER TABLE nyc.permits
CREATE BRANCH etl_job_42
""".trimIndent()).show(100, 0, false)

# Write

Before writing to the table we set `spark.wap.branch` so that writes (and reads) are against the specified branch of the table. 

In [ ]:
spark.conf().set("spark.wap.branch", "etl_job_42")

Now make the change to the table

In [ ]:
spark.sql("""
DELETE FROM nyc.permits
WHERE borough='Manhattan'
""".trimIndent()).show(100, 0, false)

## Inspecting the staged/unpublished data

### Staged/unpublished data

The changes are reflected in the table:

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits
GROUP BY borough
""".trimIndent()).show(100, 0, false)

Note that because `spark.wap.branch` is set the above query is effectively the same as this one with `VERSION AS OF` for the branch

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits VERSION AS OF 'etl_job_42'
GROUP BY borough
""".trimIndent()).show(100, 0, false)

Another syntax (albiet less clear IMHO) for `VERSION AS OF` is a `branch_<branch_name>` suffix to the table: 

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits.branch_etl_job_42
GROUP BY borough
""".trimIndent()).show(100, 0, false)

### Published data

We can also inspect the unmodified `main` version of the table with `VERSION AS OF`: 

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits VERSION AS OF 'main'
GROUP BY borough
""".trimIndent()).show(100, 0, false)

The same `branch_` suffix words here too: 

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits.branch_main
GROUP BY borough
""".trimIndent()).show(100, 0, false)

Any other user of the table will see the full set of data. We can reassure ourselves of this by unsetting `spark.wap.branch` for the session and querying the table without any `VERSION AS OF` modifier

In [ ]:
spark.conf().unset("spark.wap.branch")

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits
GROUP BY borough
""".trimIndent()).show(100, 0, false)

# Audit

How you audit the data is up to you. The nice thing about the data being staged is that you can do it within the same ETL job, or have another tool do it. 

Here's a very simple example of doing this in Kotlin. We're going to programatically check that only the four expected boroughs remain in the data. 

First, we define those that are expected: 

In [ ]:
val expected_boroughs = setOf("Queens", "Brooklyn", "Bronx", "Staten Island")

Then we get a set of the actual boroughs in the staged data

In [ ]:
val boroughs = spark.read()
    .option("branch", "etl_job_42")
    .format("iceberg")
    .load("nyc.permits")
    .select("borough")
    .distinct()
    .collect()
    .mapNotNull { it.getString(0) }
    .toSet()

Now we do two checks: 

1. Compare the length of the expected vs actual set
2. Check that the two sets when unioned are still the same length. This is necessary, since the first test isn't sufficient alone

In [ ]:
require(boroughs == expected_boroughs) {
    "Audit failed, borough set does not match expected boroughs: $boroughs != $expected_boroughs"
}
println("Audit has passed")

# Publish

Iceberg supports fast-forward merging of branches back to `main`, using the [`manageSnapshots().fastForwardBranch`](https://iceberg.apache.org/javadoc/latest/org/apache/iceberg/ManageSnapshots.html#fastForwardBranch-java.lang.String-java.lang.String-) API.

This isn't yet exposed in Spark, so the existing [`cherrypick`](https://iceberg.apache.org/javadoc/latest/org/apache/iceberg/ManageSnapshots.html#cherrypick-long-) can be used as a slightly less elegant option.

ℹ️ Note that `cherrypick` only works for one commit. 

First, we need the snapshot ID of our branch, which we can get from the `.refs` table:

In [ ]:
spark.sql("""
SELECT * FROM nyc.permits.refs 
""".trimIndent()).show(100, 0, false)

In [ ]:
val query = """
SELECT snapshot_id
FROM nyc.permits.refs
WHERE name = 'etl_job_42'
""".trimIndent()

val wap_snapshot_id = spark.sql(query).head().getAs<Long>("snapshot_id")

Now we do the publish, using `cherrypick_snapshot` and the snapshot id:

In [ ]:
val publish_query = "CALL system.cherrypick_snapshot('nyc.permits', $wap_snapshot_id)"
spark.sql(publish_query).show(100, 0, false)

Finally, we look at the table and revel in the glory that is our published changes 🎉

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits.branch_etl_job_42
GROUP BY borough
""".trimIndent()).show(100, 0, false)

We can also inspect the unmodified `main` version of the table with `VERSION AS OF`: 

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits VERSION AS OF 'main'
GROUP BY borough
""".trimIndent()).show(100, 0, false)

---

# What if You Don't Want to Publish Changes?

If you don't want to merge the branch you can simply `DROP` it. 

## Create a new branch

In [ ]:
spark.sql("""
ALTER TABLE nyc.permits
CREATE BRANCH new_etl_job
""".trimIndent()).show(100, 0, false)

## Set `spark.wap.branch`

In [ ]:
spark.conf().set("spark.wap.branch", "new_etl_job")

## Write

In [ ]:
spark.sql("""
DELETE FROM nyc.permits WHERE borough LIKE '%'
""".trimIndent()).show(100, 0, false)

## Audit

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits 
GROUP BY borough
""".trimIndent()).show(100, 0, false)

### Whoops 🤭 
We deleted all the data

### Reassure ourselves that the data is still there in `main`

In [ ]:
spark.sql("""
SELECT borough, count(*) permit_cnt
FROM nyc.permits VERSION AS OF 'main'
GROUP BY borough
""".trimIndent()).show(100, 0, false)

## Abandon changes

In [ ]:
spark.sql("""
ALTER TABLE nyc.permits
DROP BRANCH new_etl_job
""".trimIndent()).show(100, 0, false)

---

# Where Next?

For more information about write-audit-publish see [this talk from Michelle Winters](https://www.youtube.com/watch?v=fXHdeBnpXrg&t=1001s) and [this talk from Sam Redai](https://www.dremio.com/wp-content/uploads/2022/05/Sam-Redai-The-Write-Audit-Publish-Pattern-via-Apache-Iceberg.pdf).